# Build sample-level wide somatic features

Convert the compiled long-format PROFILE somatic tables into a binary wide matrix while retaining **every genomic specimen**. The output grain is one row per `(DFCI_MRN, UNIQUE_SAMPLE_ID)`. No treatment-date or one-sample-per-patient filtering occurs here.

Feature conventions mirror the Clinical Text Embedding project:

- SNVs: `GENE_SNV`
- CNVs: `GENE_AMP` and `GENE_DEL`
- SVs: dominant `GENE_PARTNER_FUSION`, `GENE_OTHER_SV`, or `GENE_SV`

`SAMPLE_COLLECTION_DT`, `TEST_ORDER_DT`, and `REPORT_DT` remain attached to each sample for later leakage-safe temporal selection. Binary zero means no reported alteration for that sample; `PANEL_VERSION` and `TEST_TYPE` are retained so downstream analyses can account for assay coverage.

In [ ]:
import os
from collections import defaultdict
from pathlib import Path

import polars as pl


PROFILE_DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/PROFILE_DATA')
FINAL_INPUT_DIR = PROFILE_DATA_PATH / 'FINAL'
OUTPUT_DIR = FINAL_INPUT_DIR

SPECIMEN_FILE = FINAL_INPUT_DIR / 'GENOMIC_SPECIMEN.parquet'
SNV_FILE = FINAL_INPUT_DIR / 'SNV.parquet'
CNV_FILE = FINAL_INPUT_DIR / 'CNV.parquet'
SV_FILE = FINAL_INPUT_DIR / 'SV.parquet'

WIDE_OUTPUT_FILE = OUTPUT_DIR / 'SOMATIC_WIDE_BY_SAMPLE.parquet'
MANIFEST_OUTPUT_FILE = OUTPUT_DIR / 'SOMATIC_FEATURE_MANIFEST.parquet'
QC_OUTPUT_FILE = OUTPUT_DIR / 'SOMATIC_WIDE_QC.parquet'

OVERWRITE = False
SAMPLE_KEY = ['DFCI_MRN', 'UNIQUE_SAMPLE_ID']
DATE_COLUMNS = ['SAMPLE_COLLECTION_DT', 'TEST_ORDER_DT', 'REPORT_DT']

SV_DOMINANCE_THRESHOLD = 0.50
SV_MIN_CASES = 20
SV_MIN_POSITIVE_SAMPLES = 10

# Values are normalized with strip + uppercase before matching. PROFILE uses
# LA/HA for low-/high-level amplification and 1DEL/2DEL for one-/two-copy
# deletion. Extend these sets deliberately if the audit finds another code.
CNV_AMP_VALUES = {
    'AMP', 'AMPLIFICATION', 'AMPLIFIED', 'GAIN', 'LA', 'HA',
    'COPY NUMBER GAIN', 'HIGH-LEVEL AMPLIFICATION',
}
CNV_DEL_VALUES = {
    'DEL', 'DELETION', 'DELETED', 'LOSS', 'HOMDEL', '1DEL', '2DEL',
    'HOMOZYGOUS DELETION', 'COPY NUMBER LOSS',
}
STRICT_CNV_TYPES = True


In [ ]:
def require_columns(path, required_columns):
    if not path.is_file():
        raise FileNotFoundError(f'Missing required input: {path}')
    names = pl.scan_parquet(path).collect_schema().names()
    missing = [column for column in required_columns if column not in names]
    if missing:
        raise RuntimeError(f'{path.name} is missing required columns: {missing}')
    return names


def normalized_gene(column):
    return (
        pl.col(column)
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase()
        .replace('', None)
    )


def events_to_wide(events, sample_keys):
    """Pivot unique SAMPLE_KEY/FEATURE events into UInt8 indicators."""
    if events.is_empty():
        return sample_keys.clone()

    events = events.select(SAMPLE_KEY + ['FEATURE']).unique()
    wide = (
        events
        .with_columns(pl.lit(1, dtype=pl.UInt8).alias('PRESENT'))
        .pivot(
            on='FEATURE',
            index=SAMPLE_KEY,
            values='PRESENT',
            aggregate_function='max',
        )
    )
    feature_columns = [column for column in wide.columns if column not in SAMPLE_KEY]
    return wide.with_columns(
        [pl.col(column).fill_null(0).cast(pl.UInt8) for column in feature_columns]
    )


def write_parquet_atomic(dataframe, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists() and not OVERWRITE:
        raise FileExistsError(
            f'Output already exists: {output_path}. Set OVERWRITE=True to replace it.'
        )

    temporary_path = output_path.with_name(f'.{output_path.name}.tmp')
    temporary_path.unlink(missing_ok=True)
    try:
        dataframe.write_parquet(
            temporary_path,
            compression='zstd',
            compression_level=9,
            statistics=True,
        )
        written = pl.scan_parquet(temporary_path)
        written_rows = written.select(pl.len()).collect(engine='streaming').item()
        if written_rows != dataframe.height:
            raise RuntimeError(
                f'{output_path.name}: expected {dataframe.height:,} rows, '
                f'found {written_rows:,} after writing.'
            )
        os.replace(temporary_path, output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise


def human_size(path):
    size = float(path.stat().st_size)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if size < 1024 or unit == 'TB':
            return f'{size:.1f} {unit}'
        size /= 1024


## 1. Establish the specimen universe

The specimen table controls the output row set. Null or duplicated sample keys stop the pipeline rather than being silently removed.

In [ ]:
specimen_columns = require_columns(
    SPECIMEN_FILE,
    SAMPLE_KEY + ['SAMPLE_ACCESSION_NBR'] + DATE_COLUMNS,
)
for path, required in (
    (SNV_FILE, SAMPLE_KEY + ['GENE']),
    (CNV_FILE, SAMPLE_KEY + ['GENE', 'CNV_TYPE_CD']),
    (SV_FILE, SAMPLE_KEY + ['PARTNER1_HUGO_GENE_NM', 'PARTNER2_HUGO_GENE_NM']),
):
    require_columns(path, required)

metadata_candidates = [
    'DFCI_MRN',
    'UNIQUE_SAMPLE_ID',
    'SAMPLE_ACCESSION_NBR',
    'SAMPLE_COLLECTION_DT',
    'TEST_ORDER_DT',
    'REPORT_DT',
    'TEST_TYPE',
    'PANEL_VERSION',
    'CANCER_TYPE',
    'PRIMARY_CANCER_DIAGNOSIS',
    'MUTATIONAL_BURDEN',
    'MISMATCH_REPAIR_STATUS',
]
metadata_columns = [column for column in metadata_candidates if column in specimen_columns]

specimens = (
    pl.scan_parquet(SPECIMEN_FILE)
    .select(metadata_columns)
    .with_columns(
        [
            pl.col(column)
            .cast(pl.String)
            .str.strptime(pl.Date, strict=False)
            .alias(column)
            for column in DATE_COLUMNS
        ]
    )
    .collect(engine='streaming')
)

null_key_rows = specimens.filter(
    pl.any_horizontal([pl.col(column).is_null() for column in SAMPLE_KEY])
)
if null_key_rows.height:
    raise RuntimeError(
        f'GENOMIC_SPECIMEN contains {null_key_rows.height:,} rows with null sample keys.'
    )

duplicate_keys = (
    specimens
    .group_by(SAMPLE_KEY)
    .len()
    .filter(pl.col('len') > 1)
)
if duplicate_keys.height:
    raise RuntimeError(
        f'GENOMIC_SPECIMEN contains {duplicate_keys.height:,} duplicated sample keys.'
    )

sample_keys = specimens.select(SAMPLE_KEY)
print(f'Specimens: {specimens.height:,}')
print(f'Patients:  {specimens.select(pl.col("DFCI_MRN").n_unique()).item():,}')


## 2. SNV carrier features

Each reported sample–gene combination becomes one `GENE_SNV` indicator. Exact duplicate calls do not change the binary value.

In [ ]:
snv_source_rows = (
    pl.scan_parquet(SNV_FILE)
    .select(pl.len().alias('n'))
    .collect(engine='streaming')
    .item()
)

snv_events_all = (
    pl.scan_parquet(SNV_FILE)
    .select(SAMPLE_KEY + ['GENE'])
    .with_columns(normalized_gene('GENE').alias('GENE'))
    .filter(
        pl.all_horizontal([pl.col(column).is_not_null() for column in SAMPLE_KEY + ['GENE']])
    )
    .with_columns(pl.concat_str([pl.col('GENE'), pl.lit('SNV')], separator='_').alias('FEATURE'))
    .select(SAMPLE_KEY + ['GENE', 'FEATURE'])
    .unique()
    .collect(engine='streaming')
)
snv_orphan_events = snv_events_all.join(sample_keys, on=SAMPLE_KEY, how='anti').height
snv_events = snv_events_all.join(sample_keys, on=SAMPLE_KEY, how='semi')
snv_wide = events_to_wide(snv_events, sample_keys)
snv_feature_columns = [column for column in snv_wide.columns if column not in SAMPLE_KEY]
print(f'SNV source rows: {snv_source_rows:,}')
print(f'Unique sample-gene events: {snv_events.height:,}')
print(f'SNV features: {len(snv_feature_columns):,}')


## 3. Amplification and deletion features

`CNV_TYPE_CD` is normalized through explicit configured value sets. With strict mode enabled, any non-null unmapped category stops the run and is printed for review.

In [ ]:
cnv_source_all = (
    pl.scan_parquet(CNV_FILE)
    .select(SAMPLE_KEY + ['GENE', 'CNV_TYPE_CD'])
    .with_columns(
        normalized_gene('GENE').alias('GENE'),
        pl.col('CNV_TYPE_CD')
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase()
        .replace('', None)
        .alias('_CNV_SOURCE_TYPE'),
    )
    .with_columns(
        pl.when(pl.col('_CNV_SOURCE_TYPE').is_in(sorted(CNV_AMP_VALUES)))
        .then(pl.lit('AMP'))
        .when(pl.col('_CNV_SOURCE_TYPE').is_in(sorted(CNV_DEL_VALUES)))
        .then(pl.lit('DEL'))
        .otherwise(None)
        .alias('_CNV_CLASS')
    )
    .collect(engine='streaming')
)
cnv_orphan_rows = cnv_source_all.join(sample_keys, on=SAMPLE_KEY, how='anti').height
cnv_source = cnv_source_all.join(sample_keys, on=SAMPLE_KEY, how='semi')

cnv_type_audit = (
    cnv_source
    .group_by(['_CNV_SOURCE_TYPE', '_CNV_CLASS'])
    .len()
    .sort('len', descending=True)
)
display(cnv_type_audit)

unmapped_cnv = cnv_source.filter(
    pl.col('_CNV_SOURCE_TYPE').is_not_null() & pl.col('_CNV_CLASS').is_null()
)
if STRICT_CNV_TYPES and unmapped_cnv.height:
    unknown_values = (
        unmapped_cnv
        .group_by('_CNV_SOURCE_TYPE')
        .len()
        .sort('len', descending=True)
        .to_dicts()
    )
    raise RuntimeError(f'Unmapped CNV_TYPE_CD values: {unknown_values}')

cnv_events = (
    cnv_source
    .filter(
        pl.all_horizontal(
            [pl.col(column).is_not_null() for column in SAMPLE_KEY + ['GENE', '_CNV_CLASS']]
        )
    )
    .with_columns(
        pl.concat_str([pl.col('GENE'), pl.col('_CNV_CLASS')], separator='_').alias('FEATURE')
    )
    .select(SAMPLE_KEY + ['GENE', '_CNV_CLASS', 'FEATURE'])
    .unique()
)
cnv_wide = events_to_wide(cnv_events, sample_keys)
cnv_feature_columns = [column for column in cnv_wide.columns if column not in SAMPLE_KEY]
print(f'CNV source rows: {cnv_source.height:,}')
print(f'Unique sample-gene-class events: {cnv_events.height:,}')
print(f'AMP/DEL features: {len(cnv_feature_columns):,}')


## 4. Structural variant and fusion features

Gene pairs are canonicalized alphabetically. Dominant partners follow the reference thresholds; less common SV features are removed after feature construction. Thresholds use unique samples, not raw event rows.

In [ ]:
sv_source_rows = (
    pl.scan_parquet(SV_FILE)
    .select(pl.len().alias('n'))
    .collect(engine='streaming')
    .item()
)

sv_pairs_all = (
    pl.scan_parquet(SV_FILE)
    .select(SAMPLE_KEY + ['PARTNER1_HUGO_GENE_NM', 'PARTNER2_HUGO_GENE_NM'])
    .with_columns(
        normalized_gene('PARTNER1_HUGO_GENE_NM').alias('_LEFT_GENE'),
        normalized_gene('PARTNER2_HUGO_GENE_NM').alias('_RIGHT_GENE'),
    )
    .filter(
        pl.all_horizontal(
            [pl.col(column).is_not_null() for column in SAMPLE_KEY + ['_LEFT_GENE', '_RIGHT_GENE']]
        )
    )
    .with_columns(
        pl.when(pl.col('_LEFT_GENE') <= pl.col('_RIGHT_GENE'))
        .then(pl.col('_LEFT_GENE'))
        .otherwise(pl.col('_RIGHT_GENE'))
        .alias('GENE_A'),
        pl.when(pl.col('_LEFT_GENE') <= pl.col('_RIGHT_GENE'))
        .then(pl.col('_RIGHT_GENE'))
        .otherwise(pl.col('_LEFT_GENE'))
        .alias('GENE_B'),
    )
    .select(SAMPLE_KEY + ['GENE_A', 'GENE_B'])
    .unique()
    .collect(engine='streaming')
)
sv_orphan_pairs = sv_pairs_all.join(sample_keys, on=SAMPLE_KEY, how='anti').height
sv_pairs = sv_pairs_all.join(sample_keys, on=SAMPLE_KEY, how='semi')

pair_samples = defaultdict(set)
gene_samples = defaultdict(set)
for row in sv_pairs.iter_rows(named=True):
    sample = tuple(row[column] for column in SAMPLE_KEY)
    gene_a = row['GENE_A']
    gene_b = row['GENE_B']
    pair_samples[(gene_a, gene_b)].add(sample)
    gene_samples[gene_a].add(sample)
    gene_samples[gene_b].add(sample)

directional_partner_samples = {}
for (gene_a, gene_b), samples in pair_samples.items():
    directional_partner_samples[(gene_a, gene_b)] = samples
    directional_partner_samples[(gene_b, gene_a)] = samples

gene_partner_totals = defaultdict(int)
for (gene, _partner), samples in directional_partner_samples.items():
    # Match the reference workflow: sum unique-sample counts across
    # partners rather than taking the union across a gene's partners.
    gene_partner_totals[gene] += len(samples)

major_partners = set()
for (gene, partner), samples in directional_partner_samples.items():
    fraction = len(samples) / gene_partner_totals[gene]
    if fraction >= SV_DOMINANCE_THRESHOLD and len(samples) >= SV_MIN_CASES:
        major_partners.add((gene, partner))

genes_with_major = {gene for gene, _ in major_partners}
sv_feature_samples = defaultdict(set)
sv_feature_metadata = {}

for gene, partner in sorted(major_partners):
    fusion_feature = f'{gene}_{partner}_FUSION'
    other_feature = f'{gene}_OTHER_SV'
    major_samples = directional_partner_samples[(gene, partner)]
    sv_feature_samples[fusion_feature].update(major_samples)
    sv_feature_samples[other_feature].update(gene_samples[gene] - major_samples)
    sv_feature_metadata[fusion_feature] = {
        'ALTERATION_CLASS': 'FUSION',
        'GENE': gene,
        'PARTNER': partner,
        'RULE': 'DOMINANT_PARTNER',
    }
    sv_feature_metadata[other_feature] = {
        'ALTERATION_CLASS': 'SV',
        'GENE': gene,
        'PARTNER': None,
        'RULE': 'NON_DOMINANT_PARTNER_EVENT',
    }

for gene in sorted(set(gene_samples) - genes_with_major):
    feature = f'{gene}_SV'
    sv_feature_samples[feature].update(gene_samples[gene])
    sv_feature_metadata[feature] = {
        'ALTERATION_CLASS': 'SV',
        'GENE': gene,
        'PARTNER': None,
        'RULE': 'GENE_LEVEL_NO_DOMINANT_PARTNER',
    }

sv_feature_samples = {
    feature: samples
    for feature, samples in sv_feature_samples.items()
    if len(samples) >= SV_MIN_POSITIVE_SAMPLES
}
sv_feature_metadata = {
    feature: sv_feature_metadata[feature]
    for feature in sv_feature_samples
}

sv_event_records = [
    {SAMPLE_KEY[0]: sample[0], SAMPLE_KEY[1]: sample[1], 'FEATURE': feature}
    for feature, samples in sv_feature_samples.items()
    for sample in samples
]
if sv_event_records:
    sv_events = pl.DataFrame(sv_event_records).select(SAMPLE_KEY + ['FEATURE'])
else:
    sv_events = sample_keys.head(0).with_columns(pl.lit(None, dtype=pl.String).alias('FEATURE'))

sv_wide = events_to_wide(sv_events, sample_keys)
sv_feature_columns = [column for column in sv_wide.columns if column not in SAMPLE_KEY]
print(f'SV source rows: {sv_source_rows:,}')
print(f'Canonical sample-pair events: {sv_pairs.height:,}')
print(f'Dominant directed gene-partner relationships: {len(major_partners):,}')
print(f'SV/fusion features retained: {len(sv_feature_columns):,}')


## 5. Assemble and validate the wide matrix

All feature tables are left-joined to the specimen universe. This guarantees that samples with no reported alteration remain present.

In [ ]:
wide_somatic = specimens
for feature_matrix in (snv_wide, cnv_wide, sv_wide):
    before_rows = wide_somatic.height
    wide_somatic = wide_somatic.join(feature_matrix, on=SAMPLE_KEY, how='left')
    if wide_somatic.height != before_rows:
        raise RuntimeError('A feature join changed the specimen row count.')

feature_columns = [
    column
    for column in wide_somatic.columns
    if column.endswith(('_SNV', '_AMP', '_DEL', '_SV', '_FUSION'))
]
if len(feature_columns) != len(set(feature_columns)):
    raise RuntimeError('Duplicate feature names were generated.')

wide_somatic = (
    wide_somatic
    .with_columns(
        [pl.col(column).fill_null(0).cast(pl.UInt8) for column in feature_columns]
    )
    .select(metadata_columns + sorted(feature_columns))
    .sort(['DFCI_MRN', 'REPORT_DT', 'TEST_ORDER_DT', 'SAMPLE_COLLECTION_DT', 'UNIQUE_SAMPLE_ID'])
)

if wide_somatic.height != specimens.height:
    raise RuntimeError(
        f'Expected {specimens.height:,} specimen rows; found {wide_somatic.height:,}.'
    )
if wide_somatic.select(pl.struct(SAMPLE_KEY).n_unique()).item() != wide_somatic.height:
    raise RuntimeError('Final sample keys are not unique.')
if feature_columns:
    invalid_binary_values = (
        wide_somatic
        .select(
            pl.any_horizontal(
                [~pl.col(column).is_in([0, 1]) for column in feature_columns]
            ).alias('invalid')
        )
        .get_column('invalid')
        .any()
    )
    if invalid_binary_values:
        raise RuntimeError('At least one feature contains a value other than 0 or 1.')

print(f'Final rows: {wide_somatic.height:,}')
print(f'Final metadata columns: {len(metadata_columns):,}')
print(f'Final binary features: {len(feature_columns):,}')


## 6. Build the feature manifest and QC report, then write outputs

In [ ]:
manifest_records = []

snv_positive_counts = snv_events.group_by(['FEATURE', 'GENE']).len()
for row in snv_positive_counts.iter_rows(named=True):
    manifest_records.append({
        'FEATURE': row['FEATURE'],
        'ALTERATION_CLASS': 'SNV',
        'GENE': row['GENE'],
        'PARTNER': None,
        'RULE': 'ANY_REPORTED_SOMATIC_CALL',
        'POSITIVE_SAMPLES': row['len'],
    })

cnv_positive_counts = cnv_events.group_by(['FEATURE', 'GENE', '_CNV_CLASS']).len()
for row in cnv_positive_counts.iter_rows(named=True):
    manifest_records.append({
        'FEATURE': row['FEATURE'],
        'ALTERATION_CLASS': row['_CNV_CLASS'],
        'GENE': row['GENE'],
        'PARTNER': None,
        'RULE': 'NORMALIZED_CNV_TYPE',
        'POSITIVE_SAMPLES': row['len'],
    })

for feature, samples in sv_feature_samples.items():
    manifest_records.append({
        'FEATURE': feature,
        **sv_feature_metadata[feature],
        'POSITIVE_SAMPLES': len(samples),
    })

manifest = pl.DataFrame(
    manifest_records,
    schema={
        'FEATURE': pl.String,
        'ALTERATION_CLASS': pl.String,
        'GENE': pl.String,
        'PARTNER': pl.String,
        'RULE': pl.String,
        'POSITIVE_SAMPLES': pl.UInt64,
    },
    strict=False,
).sort(['ALTERATION_CLASS', 'FEATURE'])

manifest_features = set(manifest.get_column('FEATURE').to_list())
if manifest_features != set(feature_columns):
    missing_from_manifest = sorted(set(feature_columns) - manifest_features)
    missing_from_wide = sorted(manifest_features - set(feature_columns))
    raise RuntimeError(
        'Manifest/wide feature mismatch. '
        f'Missing from manifest: {missing_from_manifest}; '
        f'missing from wide: {missing_from_wide}'
    )

wide_positive_counts = (
    wide_somatic
    .select([pl.col(feature).sum().alias(feature) for feature in feature_columns])
    .row(0, named=True)
    if feature_columns
    else {}
)
manifest_positive_counts = dict(
    zip(
        manifest.get_column('FEATURE').to_list(),
        manifest.get_column('POSITIVE_SAMPLES').to_list(),
    )
)
count_mismatches = {
    feature: (manifest_positive_counts[feature], wide_positive_counts[feature])
    for feature in feature_columns
    if manifest_positive_counts[feature] != wide_positive_counts[feature]
}
if count_mismatches:
    raise RuntimeError(
        f'Long-to-wide positive-count validation failed: {count_mismatches}'
    )

samples_per_patient = specimens.group_by('DFCI_MRN').len().get_column('len')
patients_with_multiple_samples = (samples_per_patient > 1).sum()
qc_records = [
    ('specimen_rows', specimens.height),
    ('unique_patients', specimens.select(pl.col('DFCI_MRN').n_unique()).item()),
    ('patients_with_multiple_samples', patients_with_multiple_samples),
    ('maximum_samples_per_patient', samples_per_patient.max()),
    ('snv_source_rows', snv_source_rows),
    ('cnv_source_rows', cnv_source.height),
    ('sv_source_rows', sv_source_rows),
    ('snv_unique_sample_gene_events', snv_events.height),
    ('cnv_unique_sample_gene_class_events', cnv_events.height),
    ('sv_unique_sample_pair_events', sv_pairs.height),
    ('snv_events_without_specimen', snv_orphan_events),
    ('cnv_rows_without_specimen', cnv_orphan_rows),
    ('sv_pairs_without_specimen', sv_orphan_pairs),
    ('snv_feature_count', len(snv_feature_columns)),
    ('amp_del_feature_count', len(cnv_feature_columns)),
    ('sv_fusion_feature_count', len(sv_feature_columns)),
    ('total_feature_count', len(feature_columns)),
    ('unmapped_cnv_rows', unmapped_cnv.height),
]
for date_column in DATE_COLUMNS:
    qc_records.append(
        (f'missing_{date_column.lower()}', specimens.get_column(date_column).null_count())
    )
for row in cnv_type_audit.iter_rows(named=True):
    source_type = row['_CNV_SOURCE_TYPE'] if row['_CNV_SOURCE_TYPE'] is not None else '<NULL>'
    qc_records.append((f'cnv_type::{source_type}', row['len']))

qc = pl.DataFrame(qc_records, schema=['METRIC', 'VALUE'], orient='row').with_columns(
    pl.col('METRIC').cast(pl.String),
    pl.col('VALUE').cast(pl.Int64),
).sort('METRIC')

# Write smaller validation artifacts first and the main matrix last. Each file
# is validated at its temporary path before an atomic rename.
write_parquet_atomic(manifest, MANIFEST_OUTPUT_FILE)
write_parquet_atomic(qc, QC_OUTPUT_FILE)
write_parquet_atomic(wide_somatic, WIDE_OUTPUT_FILE)

print(f'Wide matrix: {WIDE_OUTPUT_FILE} ({human_size(WIDE_OUTPUT_FILE)})')
print(f'Manifest:    {MANIFEST_OUTPUT_FILE} ({human_size(MANIFEST_OUTPUT_FILE)})')
print(f'QC report:   {QC_OUTPUT_FILE} ({human_size(QC_OUTPUT_FILE)})')


In [ ]:
summary = pl.DataFrame({
    'metric': [
        'patients',
        'samples',
        'patients_with_multiple_samples',
        'SNV_features',
        'AMP_DEL_features',
        'SV_FUSION_features',
        'total_features',
    ],
    'value': [
        specimens.select(pl.col('DFCI_MRN').n_unique()).item(),
        specimens.height,
        patients_with_multiple_samples,
        len(snv_feature_columns),
        len(cnv_feature_columns),
        len(sv_feature_columns),
        len(feature_columns),
    ],
})
display(summary)
display(manifest.group_by('ALTERATION_CLASS').agg(
    pl.len().alias('features'),
    pl.col('POSITIVE_SAMPLES').min().alias('minimum_positive_samples'),
    pl.col('POSITIVE_SAMPLES').max().alias('maximum_positive_samples'),
).sort('ALTERATION_CLASS'))
